<a href="https://colab.research.google.com/github/peacewhile/PHM-Learning-Task/blob/main/NGAFID_DATASET_MINIROCKET_EXAMPLE3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!python --version

In [ ]:
!git clone https://github.com/hyang0129/NGAFIDDATASET.git

!(cd NGAFIDDATASET ; git checkout main; git reset --hard HEAD; git pull)
!(cd NGAFIDDATASET ; pip install -r requirements.txt -q)

!pip install tsai -q

In [ ]:
!pip install --upgrade ipython -q

In [ ]:
%load_ext autoreload


In [ ]:
import sys
sys.path.append('/content/NGAFIDDATASET')

In [ ]:
%autoreload
from ngafiddataset.dataset.dataset import NGAFID_Dataset_Manager
from ngafiddataset.utils import connect_to_tpu

from tsai.basics import *
from tsai.models.MINIROCKET_Pytorch import *
from tsai.models.utils import *
import pandas as pd


# strategy = connect_to_tpu()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!tar -xzf /content/drive/MyDrive/NGAFIDDATASET/2days.tar.gz -C /content/

In [ ]:
import os

# 检查目录是否存在
print(os.path.exists('/content/2days'))

# 检查里面有什么文件
if os.path.exists('/content/2days'):
    files = os.listdir('/content/2days')
    print(f"文件数量: {len(files)}")
    print(f"前10个文件: {files[:10]}")
else:
    print("目录不存在！")

In [ ]:
# 先手动检查 flight_header.csv 是否存在
!ls /content/2days/flight_header.csv

In [ ]:
import pickle
import pandas as pd

# 加载元数据
df = pd.read_csv('/content/2days/flight_header.csv', index_col='Master Index')

# 直接加载 pkl 数据
with open('/content/2days/flight_data.pkl', 'rb') as f:
    data_dict = pickle.load(f)

print(type(data_dict))
print(f"航班数量: {len(data_dict)}")
print(f"第一条数据的key示例: {list(data_dict.keys())[:3]}")

In [ ]:
import sys
sys.path.append('/content/NGAFIDDATASET')

import numpy as np
import pandas as pd
import pickle

# 手动构建一个简单的 dm 对象
class SimpleDM:
    pass

dm = SimpleDM()

# 注入元数据
dm.flight_header_df = pd.read_csv(
    '/content/2days/flight_header.csv',
    index_col='Master Index'
)

# 注入时序数据
with open('/content/2days/flight_data.pkl', 'rb') as f:
    dm.data_dict = pickle.load(f)

# 加载归一化参数
stats = pd.read_csv('/content/2days/stats.csv', index_col=0)
print(stats.head())
print(stats.shape)

In [ ]:
import numpy as np

# 排除 cluster 列，只保留数值列
stats_numeric = stats.drop(columns=['cluster'])

# 注入归一化参数
dm.mins = stats_numeric.loc[1].values.astype(np.float32)  # 第1行 = mins
dm.maxs = stats_numeric.loc[0].values.astype(np.float32)  # 第0行 = maxs

# 验证
print(f"特征数量: {len(dm.mins)}")
print(f"mins 示例: {dm.mins[:5]}")
print(f"maxs 示例: {dm.maxs[:5]}")
print(f"航班数量: {len(dm.data_dict)}")
print(f"元数据行数: {len(dm.flight_header_df)}")

In [ ]:
def get_numpy_dataset(self, fold, training):
    # 根据 fold 和 training 筛选对应航班
    if training:
        mask = self.flight_header_df['fold'] != fold
    else:
        mask = self.flight_header_df['fold'] == fold

    subset = self.flight_header_df[mask]

    # 收集时序数据和标签
    data = [self.data_dict[idx] for idx in subset.index]

    return {
        'data':         data,
        'target_class': subset['target_class'].values,
        'before_after': subset['before_after'].values,
        'fold':         subset['fold'].values
    }

# 绑定方法到 dm 对象
import types
dm.get_numpy_dataset = types.MethodType(get_numpy_dataset, dm)

# 验证：测试 fold=0 的数据划分
train_dict = dm.get_numpy_dataset(fold=0, training=True)
test_dict  = dm.get_numpy_dataset(fold=0, training=False)
print(f"训练集大小: {len(train_dict['data'])}")
print(f"测试集大小: {len(test_dict['data'])}")
print(f"训练集标签示例: {train_dict['target_class'][:5]}")

In [ ]:
df.head(5)

In [ ]:
number_classes = len(dm.flight_header_df['class'].unique())
number_classes

number_hierarchies = len(dm.flight_header_df['hclass'].unique())
number_hierarchies


In [ ]:
from fastai.vision import *

In [ ]:

from fastai.callback.progress import CSVLogger

# from fastai.callbacks import CSVLogger
from tqdm.autonotebook import tqdm

In [ ]:
# 查看前5个航班的数据形状
for idx in list(dm.data_dict.keys())[:5]:
    print(f"航班 {idx}: {dm.data_dict[idx].shape}")

In [ ]:
import numpy as np

# 统计所有航班时间步长
lengths = [dm.data_dict[idx].shape[0] for idx in dm.data_dict.keys()]
lengths = np.array(lengths)

print(f"最短: {lengths.min()}")
print(f"最长: {lengths.max()}")
print(f"中位数: {np.median(lengths):.0f}")
print(f"均值: {lengths.mean():.0f}")
print(f"90%分位数: {np.percentile(lengths, 90):.0f}")
print(f"10%分位数: {np.percentile(lengths, 10):.0f}")

In [ ]:
def pad_or_truncate(data_list, target_len=3000):
    result = []
    for arr in data_list:
        T = arr.shape[0]
        if T >= target_len:
            arr = arr[:target_len, :]
        else:
            pad = np.zeros((target_len - T, arr.shape[1]), dtype=np.float32)
            arr = np.concatenate([arr, pad], axis=0)
        result.append(arr)
    return np.array(result, dtype=np.float32).transpose(0, 2, 1)

In [ ]:
save_path = '.'
model_name = 'MINIROCKET_BINARY'

# 提前计算归一化参数（只取前23个）
mins23 = dm.mins[:23].reshape(-1, 1)
maxs23 = dm.maxs[:23].reshape(-1, 1)

for fold in tqdm(range(5)):

    save_filename = save_path + '%s_%i' % (model_name, fold)

    train_dict = dm.get_numpy_dataset(fold=fold, training=True)
    test_dict  = dm.get_numpy_dataset(fold=fold, training=False)

    train_X = pad_or_truncate(train_dict['data'], target_len=3000)
    train_X = (train_X - mins23) / (maxs23 - mins23)
    train_X = np.nan_to_num(train_X, copy=False)

    test_X = pad_or_truncate(test_dict['data'], target_len=3000)
    test_X = (test_X - mins23) / (maxs23 - mins23)
    test_X = np.nan_to_num(test_X, copy=False)

    train_Y = np.array(train_dict['before_after'])
    test_Y  = np.array(test_dict['before_after'])

    splits = [list(np.arange(len(train_Y)))]
    splits.append(list(np.arange(len(test_Y)) + len(train_Y)))

    torch.cuda.empty_cache()
    mrf = MiniRocketFeatures(train_X.shape[1], train_X.shape[2]).to(default_device())
    chunksize = 64

    mrf.fit(train_X, chunksize=chunksize)

    X_feat = get_minirocket_features(np.concatenate([train_X, test_X]), mrf, chunksize=chunksize, to_np=True)

    Y = np.concatenate([train_Y, test_Y])

    PATH = Path("./models/MRF.pt")
    PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(mrf.state_dict(), PATH)

    tfms = [None, TSClassification()]
    batch_tfms = TSStandardize(by_sample=True)
    dls = get_ts_dls(X_feat, Y, splits=splits, tfms=tfms, batch_tfms=batch_tfms)
    model = build_ts_model(MiniRocketHead, dls=dls)

    learn = Learner(dls, model, metrics=accuracy, cbs=[ShowGraph(), CSVLogger(save_filename)])

    results = learn.fit_one_cycle(200, 2.5e-5)